# Sensitivity Analysis - Ketidakpastian Label Negatif

Baseline model (`02_baseline_model.ipynb`) dan ablation study
(`03_ablation_study.ipynb`) memakai `is_training_eligible` - aturan
"normal" untuk menentukan snapshot mana yang boleh dianggap negatif
(tidak rusak dalam 30 hari). Sebagian snapshot negatif berasal dari cycle
yang masih berjalan (right-censored) dan PART-nya cukup lama tidak
terlihat lagi di data - kita sebenarnya tidak 100% yakin PART tersebut
benar-benar masih aman.

Notebook ini membandingkan **tiga** aturan kelayakan negatif untuk
menjawab: kalau kita cuma percaya negatif yang paling meyakinkan, apakah
kesimpulan model berubah drastis, atau tetap kurang lebih sama?

1. **Normal** (`is_training_eligible`) - seluruh negatif yang lolos aturan dasar.
2. **Strict** (`is_strict_training_eligible`) - hanya negatif dari PART yang
   masih terlihat aktif mendekati akhir data (berbasis jarak waktu).
3. **RECON-verified** (`is_recon_verified_training_eligible`) - hanya
   menolak negatif kalau ada RECON yang terbukti muncul setelah PART
   terakhir terlihat aktif, tanpa mensyaratkan aktivitas baru-baru ini.
   Aturan ini muncul dari diskusi review: RECON yang muncul belakangan
   adalah sinyal langsung "ada yang perlu direkonsiliasi", lebih tepat
   sasaran daripada sekadar "sudah berapa lama diam".

- Kalau ketiganya **relatif mirip** -> model cukup tahan terhadap
  ketidakpastian label negatif.
- Kalau **Strict jauh berbeda tapi RECON-verified tetap mirip Normal** ->
  aturan Strict yang lama terlalu kasar/berlebihan, bukan berarti datanya
  benar-benar buruk.
- Kalau **ketiganya jauh berbeda** -> kualitas label negatif memang risiko
  nyata yang perlu ditangani sebelum produksi.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from catboost import CatBoostClassifier, Pool

PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 0. Tujuan

**Tujuan:** menguji apakah baseline model (kelompok C / 16 fitur inti, model
CatBoost - kandidat utama dari `02_baseline_model.ipynb`) tetap memberi
kesimpulan yang mirip pada tiga definisi "negatif yang bisa dipercaya".

Pengaturan model dibuat sama persis untuk ketiga varian
(`auto_class_weights='Balanced'`, early stopping) - satu-satunya yang
berubah adalah baris mana yang dianggap layak dipakai training.


In [ ]:
ELIGIBILITY_VARIANTS = {
    'Normal': 'is_training_eligible',
    'Strict': 'is_strict_training_eligible',
    'RECON-verified': 'is_recon_verified_training_eligible',
}

dataset = query("""
    SELECT f.*, l.target_failure_30d, l.temporal_split,
        l.is_training_eligible, l.is_strict_training_eligible,
        l.is_recon_verified_training_eligible
    FROM analytics.failure_30d_baseline_features f
    JOIN analytics.failure_30d_model_labels l
      USING (installation_cycle_id, item_identifier_clean, observation_on)
    WHERE l.temporal_split IN ('TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026')
      AND (l.is_training_eligible OR l.is_strict_training_eligible OR l.is_recon_verified_training_eligible)
""")

categorical_features = ['part_model_category', 'client_category', 'installation_age_band']
numeric_features = [
    'log_days_since_installation', 'log_total_prior_events', 'log_prior_failure_count',
    'has_prior_failure', 'log_prior_corrective_count', 'has_prior_corrective',
    'log_days_since_last_corrective', 'log_prior_distinct_places', 'log_prior_corrective_30d',
    'log_prior_failure_365d', 'log_prior_events_180d', 'month_sin', 'month_cos',
]
feature_columns = categorical_features + numeric_features
dataset[categorical_features] = dataset[categorical_features].astype(str)
dataset[numeric_features] = dataset[numeric_features].apply(pd.to_numeric)
dataset['target_failure_30d'] = dataset['target_failure_30d'].astype(bool)

rows = []
for split_name in ['TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026']:
    part = dataset.loc[dataset.temporal_split.eq(split_name)]
    row = {'Split': split_name}
    for variant_name, column in ELIGIBILITY_VARIANTS.items():
        subset = part.loc[part[column]]
        row[f'{variant_name} - baris'] = len(subset)
        row[f'{variant_name} - positive rate (%)'] = round(100 * subset.target_failure_30d.mean(), 3) if len(subset) else 0.0
    rows.append(row)
display(pd.DataFrame(rows))

## 1. Latih model pada tiga versi data

Tiga model dilatih terpisah, satu per aturan kelayakan, dengan pengaturan
yang identik.


In [ ]:
def topk_table(y_true, y_proba, k_values):
    order = np.argsort(-y_proba)
    y_sorted = np.asarray(y_true)[order]
    total_positive = int(np.sum(y_true))
    out = []
    for k in k_values:
        k = min(k, len(y_sorted))
        caught = int(y_sorted[:k].sum())
        out.append({
            'K': k, 'Kerusakan tertangkap': caught,
            'Precision@K (%)': round(100.0 * caught / k, 2),
            'Recall@K (%)': round(100.0 * caught / total_positive, 2) if total_positive else 0.0,
        })
    return pd.DataFrame(out)


def build_split(column):
    out = {}
    for split_name in ['TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026']:
        part = dataset.loc[dataset.temporal_split.eq(split_name) & dataset[column]]
        out[split_name] = (part[feature_columns], part['target_failure_30d'])
    return out


def train_catboost(X_train, y_train, X_val, y_val):
    train_pool = Pool(X_train, y_train, cat_features=categorical_features)
    val_pool = Pool(X_val, y_val, cat_features=categorical_features)
    model = CatBoostClassifier(
        iterations=1000, depth=6, learning_rate=0.05, loss_function='Logloss',
        eval_metric='PRAUC', auto_class_weights='Balanced', random_seed=RANDOM_STATE,
        early_stopping_rounds=50, verbose=False,
    )
    model.fit(train_pool, eval_set=val_pool)
    return model


models, splits_by_variant, val_proba_by_variant, test_proba_by_variant = {}, {}, {}, {}
for variant_name, column in ELIGIBILITY_VARIANTS.items():
    splits = build_split(column)
    splits_by_variant[variant_name] = splits
    X_train, y_train = splits['TRAIN_2014_2024']
    X_val, y_val = splits['VALIDATION_2025']
    X_test, y_test = splits['TEST_2026']
    model = train_catboost(X_train, y_train, X_val, y_val)
    models[variant_name] = model
    val_proba_by_variant[variant_name] = model.predict_proba(X_val)[:, 1]
    test_proba_by_variant[variant_name] = model.predict_proba(X_test)[:, 1]
    print(f"[selesai] {variant_name}: {len(y_train):,} baris latih, PR-AUC val={average_precision_score(y_val, val_proba_by_variant[variant_name]):.4%}, ROC-AUC val={roc_auc_score(y_val, val_proba_by_variant[variant_name]):.4f}, best_iter={model.get_best_iteration()}".replace(',', '.'))

## 2. Perbandingan ringkas: PR-AUC dan ROC-AUC

**ROC-AUC lebih layak dijadikan pembanding utama di sini** dibanding
PR-AUC, karena PR-AUC secara alami naik hanya karena persentase positif
berbeda-beda antar varian data (lihat tabel bagian 0) - bukan murni karena
model lebih pintar. ROC-AUC jauh lebih tahan terhadap perbedaan itu,
sehingga lebih jujur menunjukkan apakah kemampuan model membedakan
positif/negatif benar-benar berubah.


In [ ]:
comparison_rows = []
for variant_name in ELIGIBILITY_VARIANTS:
    X_val, y_val = splits_by_variant[variant_name]['VALIDATION_2025']
    X_test, y_test = splits_by_variant[variant_name]['TEST_2026']
    comparison_rows.append({
        'Aturan': variant_name,
        'PR-AUC validasi': average_precision_score(y_val, val_proba_by_variant[variant_name]),
        'ROC-AUC validasi': roc_auc_score(y_val, val_proba_by_variant[variant_name]),
        'PR-AUC test': average_precision_score(y_test, test_proba_by_variant[variant_name]),
        'ROC-AUC test': roc_auc_score(y_test, test_proba_by_variant[variant_name]),
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison.style.format({c: '{:.4%}' for c in comparison.columns if 'PR-AUC' in c} |
                                 {c: '{:.4f}' for c in comparison.columns if 'ROC-AUC' in c}))

plt.figure(figsize=(7, 5))
sns.barplot(data=comparison, x='Aturan', y='ROC-AUC validasi', color='steelblue')
plt.ylim(0.5, 1.0)
plt.axhline(comparison.loc[comparison.Aturan.eq('Normal'), 'ROC-AUC validasi'].iloc[0],
            linestyle='--', color='gray', label='ROC-AUC Normal (acuan)')
plt.title('ROC-AUC validasi per aturan kelayakan negatif')
plt.legend(); plt.tight_layout(); plt.show()

## 3. Precision/Recall@K per aturan (data validasi)


In [ ]:
for variant_name in ELIGIBILITY_VARIANTS:
    _, y_val = splits_by_variant[variant_name]['VALIDATION_2025']
    display(Markdown(f'**{variant_name} - Precision/Recall@K (validasi)**'))
    display(topk_table(y_val, val_proba_by_variant[variant_name], [100, 500, 1000, min(5000, len(y_val))]))

plt.figure(figsize=(7, 6))
for variant_name in ELIGIBILITY_VARIANTS:
    _, y_val = splits_by_variant[variant_name]['VALIDATION_2025']
    precision, recall, _ = precision_recall_curve(y_val, val_proba_by_variant[variant_name])
    plt.plot(recall, precision, label=variant_name)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve per aturan (masing-masing di data validasinya)')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Kesimpulan


In [ ]:
normal_roc_val = comparison.loc[comparison.Aturan.eq('Normal'), 'ROC-AUC validasi'].iloc[0]
strict_roc_val = comparison.loc[comparison.Aturan.eq('Strict'), 'ROC-AUC validasi'].iloc[0]
recon_roc_val = comparison.loc[comparison.Aturan.eq('RECON-verified'), 'ROC-AUC validasi'].iloc[0]
strict_gap = abs(normal_roc_val - strict_roc_val)
recon_gap = abs(normal_roc_val - recon_roc_val)
strict_rows = len(splits_by_variant['Strict']['TRAIN_2014_2024'][1])
recon_rows = len(splits_by_variant['RECON-verified']['TRAIN_2014_2024'][1])
normal_rows = len(splits_by_variant['Normal']['TRAIN_2014_2024'][1])

verdict = (
    'Aturan Strict yang lama tampak "berubah drastis" terutama karena terlalu banyak membuang data yang sebenarnya tidak bermasalah (dibuang hanya karena sudah lama diam, bukan karena ada bukti masalah). Begitu memakai kriteria yang lebih tepat sasaran (RECON-verified), ROC-AUC kembali stabil mendekati aturan Normal.'
    if recon_gap < strict_gap / 2 else
    'Ketiga aturan tetap menunjukkan perbedaan yang cukup berarti - kualitas label negatif masih layak ditelusuri lebih lanjut bersama tim operasional.'
)

display(Markdown(f"""**Ringkasan sensitivity analysis (3 aturan)**

- **Normal**: {normal_rows:,} baris latih, ROC-AUC validasi **{normal_roc_val:.4f}**.
- **Strict**: {strict_rows:,} baris latih ({100*strict_rows/normal_rows:.1f}% dari Normal), ROC-AUC validasi **{strict_roc_val:.4f}** (selisih {strict_gap:.4f} dari Normal).
- **RECON-verified**: {recon_rows:,} baris latih ({100*recon_rows/normal_rows:.1f}% dari Normal), ROC-AUC validasi **{recon_roc_val:.4f}** (selisih {recon_gap:.4f} dari Normal).

**{verdict}**

**Rekomendasi:**

1. Pakai `is_recon_verified_training_eligible` sebagai aturan kelayakan negatif utama untuk tahap selanjutnya - lebih tepat sasaran daripada Normal maupun Strict, dan hasilnya paling stabil.
2. Tetap konfirmasikan ke tim operasional OM: PART yang akhirnya kena RECON setelah lama tidak aktif - apakah pola ini konsisten dengan proses penarikan/decommission resmi, atau ada kemungkinan kerusakan yang tidak tercatat lewat jalur corrective.
3. `is_strict_training_eligible` tetap disimpan di pipeline untuk audit/pembanding, tapi tidak lagi jadi acuan utama sensitivity analysis.""".replace(',', '.')))